# TCC - Analise Preditiva de Risco Academico Escolar
## CRISP-DM: 2. Data Understanding

## Objetivos da fase

- Listar quais arquivos CSV alimentam a pipeline.
- Distinguir bases centrais de modelagem e bases de apoio relatorial.
- Entender a granularidade de cada arquivo.
- Verificar quantidade de linhas, colunas e campos ausentes.
- Observar pequenas amostras públicas já pseudonimizadas.
- Conferir cobertura de anos letivos, series, turmas, disciplinas e professores.
- Identificar problemas que precisam ser tratados na fase seguinte, Data Preparation.

## Carregar bibliotecas

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ExecutionError: ModuleNotFoundError: No module named 'IPython'

## Localizar os CSVs canonicos

A pipeline publica do projeto nao depende diretamente do banco institucional. Ela espera arquivos CSV ja extraidos e tratados em `artifacts/database/csv/`, conforme o contrato publico documentado em `docs/ENTRADA_DE_DADOS_E_CONTRATOS.md`.

In [2]:
def discover_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "artifacts" / "database" / "csv").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = discover_project_root()
CSV_DIR = PROJECT_ROOT / "artifacts" / "database" / "csv"

EXPECTED_FILES = {
    "alunos": "aluno.csv",
    "notas": "media_nota_aluno.csv",
    "faltas": "faltas_aluno.csv",
    "pagamentos": "pagamento_aluno.csv",
    "responsaveis": "responsaveis_aluno.csv",
    "professores": "professor_disciplina.csv",
}

CSV_DIR

PosixPath('/Users/warley/Desktop/Development/Personal/KnowledgeGraphModelSchoolEducation/artifacts/database/csv')

## Carregar arquivos disponiveis

Se algum CSV nao existir localmente, ele aparece como ausente no inventario. Isso permite que o notebook seja aberto tambem por avaliadores que nao tenham acesso aos dados reais.

In [3]:
def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path, encoding="utf-8", low_memory=False)


tables = {
    name: read_csv_if_exists(CSV_DIR / filename)
    for name, filename in EXPECTED_FILES.items()
}

inventory = []
for name, filename in EXPECTED_FILES.items():
    frame = tables[name]
    inventory.append({
        "base": name,
        "arquivo": filename,
        "existe": frame is not None,
        "linhas": 0 if frame is None else len(frame),
        "colunas": 0 if frame is None else len(frame.columns),
    })

pd.DataFrame(inventory)

,base,arquivo,existe,linhas,colunas
0,alunos,aluno.csv,True,19661,26
1,notas,media_nota_aluno.csv,True,239678,20
2,faltas,faltas_aluno.csv,True,168051,22
3,pagamentos,pagamento_aluno.csv,True,220090,26
4,responsaveis,responsaveis_aluno.csv,True,39256,26
5,professores,professor_disciplina.csv,True,1624,15


### Visualmente, o que saiu desta etapa

A tabela acima e a primeira fotografia real do dado dentro do notebook. Ela mostra, de forma objetiva, quais CSVs existem, o tamanho de cada base e quais arquivos ainda estariam faltando para o restante da pipeline.

### Exemplo visual fixo do inventário

Abaixo está um exemplo estático do que a célula acima entrega quando o dado é carregado:

| base | arquivo | existe | linhas | colunas |
| --- | --- | --- | --- | --- |
| alunos | aluno.csv | True | 19661 | 26 |
| notas | media_nota_aluno.csv | True | 239678 | 20 |
| faltas | faltas_aluno.csv | True | 168051 | 22 |
| pagamentos | pagamento_aluno.csv | True | 220090 | 26 |
| professores | professor_disciplina.csv | True | 1624 | 15 |
| responsaveis | responsaveis_aluno.csv | True | 39256 | 26 |

In [4]:
inventory_frame = pd.DataFrame(inventory).sort_values("base").reset_index(drop=True)
inventory_frame

,base,arquivo,existe,linhas,colunas
0,alunos,aluno.csv,True,19661,26
1,faltas,faltas_aluno.csv,True,168051,22
2,notas,media_nota_aluno.csv,True,239678,20
3,pagamentos,pagamento_aluno.csv,True,220090,26
4,professores,professor_disciplina.csv,True,1624,15
5,responsaveis,responsaveis_aluno.csv,True,39256,26


Aqui o dado ainda nao foi transformado. Ele apenas foi localizado e carregado. O ganho desta etapa e enxergar a disponibilidade real das bases antes de qualquer análise mais profunda.

### O que aconteceu com os dados apos esta celula

Quando voce executar a celula acima, ela nao altera nenhum CSV. Ela apenas monta um inventario seguro dizendo quais arquivos existem, quantas linhas possuem e quantas colunas carregam.

A leitura didatica aqui e simples:
- se um arquivo central estiver ausente, a pipeline nao conseguira montar o dataset de modelagem completo;
- se um arquivo opcional ou de apoio estiver ausente, o notebook ainda abre, mas a camada analitica correspondente fica limitada;
- diferencas grandes de volume entre bases ja sugerem, desde cedo, que sera necessario agregar e alinhar granularidades na fase seguinte.

## Dicionario conceitual dos arquivos

| Base | Arquivo | Papel na pipeline | Granularidade esperada |
|---|---|---|---|
| Alunos | `aluno.csv` | contexto escolar e matricula | aluno x matricula x turma x periodo |
| Notas | `media_nota_aluno.csv` | historico principal de desempenho | aluno x disciplina x etapa |
| Faltas | `faltas_aluno.csv` | sinal complementar de frequencia | evento de falta por aluno x disciplina x etapa |
| Pagamentos | `pagamento_aluno.csv` | contexto financeiro agregado | movimento financeiro por matricula |
| Responsaveis | `responsaveis_aluno.csv` | apoio cadastral e relatorial | aluno x responsavel |
| Professores | `professor_disciplina.csv` | vinculo pedagogico por turma e disciplina | turma x disciplina x professor |

Na arquitetura atual, o nucleo da modelagem usa diretamente `notas`, `faltas`, `pagamentos` e `professores`. As bases `aluno` e `responsaveis` funcionam principalmente como apoio contextual, documental e relatorial. A fase de entendimento dos dados verifica se esses arquivos existem, se possuem as chaves minimas e se podem ser integrados sem descartar alunos, turmas ou disciplinas sem professor vinculado.

## Funcoes para amostra publica

A base pública deste projeto já passa por tratamento para que pessoas reais não sejam identificadas. Por isso, as amostras abaixo podem usar os nomes pseudonimizados do próprio CSV, como `Aluno 00418` e `Professor 00207`, sem precisar aplicar uma segunda mascarização no notebook.

In [ ]:
def public_preview(frame: pd.DataFrame, rows: int = 5, max_columns: int = 12) -> pd.DataFrame:
    preview = frame.head(rows).copy()
    if len(preview.columns) > max_columns:
        preview = preview.iloc[:, :max_columns]
    return preview.fillna("<ausente>")


for name, frame in tables.items():
    if frame is not None:
        print(f"\n{name} - amostra publica")
        display(public_preview(frame))

### Pequena amostra visual do que chegou em cada base

Assim como no notebook de referência, a saída acima deixa claro como os dados "parecem" depois da leitura. Como esta versão do projeto já trabalha com CSVs públicos tratados, o notebook pode mostrar os nomes pseudonimizados da própria base sem precisar trocá-los por hashes artificiais.

In [6]:
preview_catalog = []
for name, frame in tables.items():
    if frame is None or frame.empty:
        continue
    preview = public_preview(frame, rows=2, max_columns=6)
    preview_catalog.append({
        "base": name,
        "linhas_mostradas": len(preview),
        "colunas_mostradas": ", ".join(preview.columns.astype(str).tolist()),
    })

pd.DataFrame(preview_catalog)

,base,linhas_mostradas,colunas_mostradas
0,alunos,2,"IDUnidade, NomeUnidade, IDUnidadexTipoCurso, IDPeriodo, NomePeriodo, IDCursoxPeriodo"
1,notas,2,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso"
2,faltas,2,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso"
3,pagamentos,2,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso"
4,responsaveis,2,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso"
5,professores,2,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso"


Esse quadro-resumo funciona como legenda da amostra: ele diz exatamente qual recorte de cada base foi exibido logo acima.

### O que observar nas amostras pseudonimizadas

A celula acima mostra a forma dos dados sem expor pessoas reais. O objetivo nao e inspecionar nomes ou IDs, mas enxergar o desenho da base: quais colunas aparecem juntas, que tipos de valores existem e se a granularidade bate com o contrato esperado.

Quando a amostra estiver correta, voce deve conseguir reconhecer o papel da base sem precisar ver nenhum dado sensivel.

## Colunas disponiveis por base

Este bloco ajuda a verificar se os CSVs seguem o contrato esperado. Ele mostra apenas nomes de colunas, nao dados reais.

### Exemplo visual fixo das colunas principais

| base | colunas_chave |
| --- | --- |
| notas | IDAluno, NomePeriodo, NomeDisciplina, NomeEtapa, ValorMedia |
| faltas | IDAluno, NomePeriodo, NomeDisciplina, NomeEtapa, DataFalta |
| pagamentos | IDAluno, NomePeriodo, PagoMovimento, EhMensalidadeMovimento, ValorPagoMovimento |
| professores | IDTurma, IDDisciplina, IDFuncionario, NomeFuncionario |

In [7]:
columns_summary = []
for name, frame in tables.items():
    if frame is None:
        columns_summary.append({"base": name, "colunas": "arquivo ausente"})
    else:
        columns_summary.append({"base": name, "colunas": ", ".join(frame.columns)})

pd.DataFrame(columns_summary)

,base,colunas
0,alunos,"IDUnidade, NomeUnidade, IDUnidadexTipoCurso, IDPeriodo, NomePeriodo, IDCursoxPeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDMatricula, SituacaoMatricula, DataMatricula, IDAluno, CodigoAluno, NomeAluno, SexoAluno, DataNascimentoAluno, QuadraResidenciaAluno, LoteResidenciaAluno, NumeroResidenciaAluno, ComplementoResidenciaAluno, BairroResidenciaAluno, CEPResidenciaAluno"
1,notas,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDMatricula, SituacaoMatricula, IDAluno, NomeAluno, IDDisciplina, NomeDisciplina, IDEtapa, NomeEtapa, IDMedia, ValorMedia"
2,faltas,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDMatricula, SituacaoMatricula, IDAluno, NomeAluno, IDDisciplinaxSerie, IDDisciplina, NomeDisciplina, IDEtapaxSerie, IDEtapa, NomeEtapa, IDFalta, DataFalta"
3,pagamentos,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDMatricula, SituacaoMatricula, IDAluno, NomeAluno, IDMovimento, IDMatriculaMovimento, ParcelaMovimento, DescricaoMovimento, DataAntecipadoMovimento, ValorAntecipadoMovimento, DataVencimentoMovimento, ValorMovimento, PagoMovimento, ValorPagoMovimento, EhMensalidadeMovimento, EhMatriculaMovimento"
4,responsaveis,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDMatricula, SituacaoMatricula, IDAluno, NomeAluno, IDResponsavel, TipoResponsavel, NomeResponsavel, SexoResponsavel, DataNascimentoResponsavel, LogradouroResidenciaResponsavel, QuadraResidenciaResponsavel, LoteResidenciaResponsavel, NumeroResidenciaResponsavel, ComplementoResidenciaResponsavel, BairroResidenciaResponsavel, CEPResidenciaResponsavel"
5,professores,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie, IDTurma, ApelidoTurma, IDDisciplinaxFuncionario, IDFuncionario, NomeFuncionario, IDDisciplina, NomeDisciplina"


### O que aconteceu com o dado apos listar as colunas

Depois desta celula, o notebook deixa de mostrar apenas exemplos de valores e passa a mostrar a estrutura formal de cada CSV. Isso ajuda a verificar se o contrato esperado foi realmente respeitado.

In [8]:
column_count_frame = pd.DataFrame([
    {
        "base": name,
        "quantidade_colunas": 0 if frame is None else len(frame.columns),
        "primeiras_colunas": "arquivo ausente" if frame is None else ", ".join(frame.columns[:8].astype(str).tolist()),
    }
    for name, frame in tables.items()
])
column_count_frame

,base,quantidade_colunas,primeiras_colunas
0,alunos,26,"IDUnidade, NomeUnidade, IDUnidadexTipoCurso, IDPeriodo, NomePeriodo, IDCursoxPeriodo, IDCurso, NomeCurso"
1,notas,20,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie"
2,faltas,22,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie"
3,pagamentos,26,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie"
4,responsaveis,26,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie"
5,professores,15,"IDUnidade, NomeUnidade, IDPeriodo, NomePeriodo, IDCurso, NomeCurso, IDSerie, NomeSerie"


### O que esta saida confirma

Aqui voce enxerga o dicionario real que chegou do ambiente local. Se alguma coluna esperada nao aparecer, o problema ainda nao e de modelagem: ele ja nasceu na extração ou no contrato do CSV.

Esse bloco e importante porque separa duas perguntas diferentes:
- o dado existe?
- o dado existe no formato que a pipeline espera?

## Qualidade inicial: valores ausentes

Valores ausentes nao sao necessariamente erro. Por exemplo, uma disciplina pode nao ter professor vinculado no banco. O importante e saber onde ha ausencia para decidir tratamento correto na proxima fase.

### Exemplo visual fixo dos ausentes

| base | coluna_exemplo | ausentes_exemplo |
| --- | --- | --- |
| notas | ValorMedia | 3405 |
| faltas | DataFalta | 0 |
| pagamentos | DataAntecipadoMovimento | 0 |
| professores | NomeFuncionario | 0 |

In [9]:
def missing_report(frame: pd.DataFrame, top: int = 10) -> pd.DataFrame:
    missing = frame.isna().sum().sort_values(ascending=False)
    report = missing.to_frame("ausentes")
    report["percentual"] = (report["ausentes"] / len(frame) * 100).round(2)
    return report.reset_index(names="coluna").head(top)


for name, frame in tables.items():
    if frame is not None and len(frame) > 0:
        print(f"\n{name} - maiores percentuais de ausencia")
        display(missing_report(frame))


alunos - maiores percentuais de ausencia

notas - maiores percentuais de ausencia

faltas - maiores percentuais de ausencia

pagamentos - maiores percentuais de ausencia

responsaveis - maiores percentuais de ausencia

professores - maiores percentuais de ausencia


,coluna,ausentes,percentual
0,CEPResidenciaAluno,19661,100.00
1,ComplementoResidenciaAluno,19661,100.00
2,NumeroResidenciaAluno,19661,100.00
3,LoteResidenciaAluno,19661,100.00
4,QuadraResidenciaAluno,19661,100.00
5,NomeCurso,985,5.01
6,NomeSerie,985,5.01
7,DataMatricula,86,0.44
8,BairroResidenciaAluno,0,0.00
9,DataNascimentoAluno,0,0.00


,coluna,ausentes,percentual
0,ValorMedia,3405,1.42
1,NomeUnidade,0,0.00
2,IDMedia,0,0.00
3,NomeEtapa,0,0.00
4,IDEtapa,0,0.00
5,NomeDisciplina,0,0.00
6,IDDisciplina,0,0.00
7,NomeAluno,0,0.00
8,IDAluno,0,0.00
9,SituacaoMatricula,0,0.00


,coluna,ausentes,percentual
0,IDUnidade,0,0.0
1,NomeUnidade,0,0.0
2,IDFalta,0,0.0
3,NomeEtapa,0,0.0
4,IDEtapa,0,0.0
5,IDEtapaxSerie,0,0.0
6,NomeDisciplina,0,0.0
7,IDDisciplina,0,0.0
8,IDDisciplinaxSerie,0,0.0
9,NomeAluno,0,0.0


,coluna,ausentes,percentual
0,ValorPagoMovimento,26949,12.24
1,NomeCurso,6502,2.95
2,NomeSerie,6502,2.95
3,IDUnidade,0,0.00
4,IDMovimento,0,0.00
5,EhMensalidadeMovimento,0,0.00
6,PagoMovimento,0,0.00
7,ValorMovimento,0,0.00
8,DataVencimentoMovimento,0,0.00
9,ValorAntecipadoMovimento,0,0.00


,coluna,ausentes,percentual
0,DataNascimentoResponsavel,36607,93.25
1,LoteResidenciaResponsavel,34568,88.06
2,QuadraResidenciaResponsavel,34140,86.97
3,ComplementoResidenciaResponsavel,33177,84.51
4,NumeroResidenciaResponsavel,20189,51.43
5,LogradouroResidenciaResponsavel,17777,45.28
6,CEPResidenciaResponsavel,17774,45.28
7,BairroResidenciaResponsavel,17765,45.25
8,NomeCurso,1951,4.97
9,NomeSerie,1951,4.97


,coluna,ausentes,percentual
0,IDUnidade,0,0.0
1,NomeUnidade,0,0.0
2,IDPeriodo,0,0.0
3,NomePeriodo,0,0.0
4,IDCurso,0,0.0
5,NomeCurso,0,0.0
6,IDSerie,0,0.0
7,NomeSerie,0,0.0
8,IDTurma,0,0.0
9,ApelidoTurma,0,0.0


### Pequena saída visual do efeito dos ausentes

A listagem acima mostra onde cada base chega mais incompleta. Para deixar isso ainda mais visível, o quadro abaixo resume o maior percentual de ausência encontrado em cada CSV.

In [10]:
missing_summary = []
for name, frame in tables.items():
    if frame is None or frame.empty:
        continue
    report = missing_report(frame, top=1)
    if report.empty:
        continue
    top_row = report.iloc[0]
    missing_summary.append({
        "base": name,
        "coluna_mais_ausente": top_row["coluna"],
        "ausentes": int(top_row["ausentes"]),
        "percentual": float(top_row["percentual"]),
    })

pd.DataFrame(missing_summary).sort_values("percentual", ascending=False)

,base,coluna_mais_ausente,ausentes,percentual
0,alunos,CEPResidenciaAluno,19661,100.00
4,responsaveis,DataNascimentoResponsavel,36607,93.25
3,pagamentos,ValorPagoMovimento,26949,12.24
1,notas,ValorMedia,3405,1.42
2,faltas,IDUnidade,0,0.00
5,professores,IDUnidade,0,0.00


### Como interpretar os ausentes

A saida acima mostra onde a base esta incompleta. Isso nao significa automaticamente erro. Em varios casos, dado ausente representa uma ausencia legitima de vinculo, como disciplina sem professor identificado.

O ponto didatico desta fase e entender que dado faltante pode significar tres coisas diferentes:
- problema de extração;
- variavel realmente opcional;
- sinal de negocio que precisara ser tratado explicitamente na preparação.

## Cobertura temporal e academica

Como o projeto faz previsao temporal, e essencial entender quais anos letivos existem, como eles se distribuem entre as bases e se ha dados suficientes para separar treino, validacao e teste por ano.

### Exemplo visual fixo da cobertura academica

Em vez de repetir linhas parecidas, a tabela abaixo resume melhor o alcance temporal de cada base:

| base | ano_inicial | ano_final | anos_distintos | disciplinas_distintas |
| --- | --- | --- | --- | --- |
| notas | 2020 | 2026 | 7 | 70 |
| faltas | 2023 | 2026 | 4 | 57 |
| pagamentos | 2004 | 2026 | 23 | <não se aplica> |
| professores | 2022 | 2026 | 5 | 66 |

In [11]:
coverage_rows = []
for name, frame in tables.items():
    if frame is None:
        continue
    row = {"base": name, "linhas": len(frame)}
    if "NomePeriodo" in frame.columns:
        periods = sorted(frame["NomePeriodo"].dropna().astype(str).unique().tolist())
        row["periodos_exemplo"] = ", ".join(periods[:6])
    for column in ["NomePeriodo", "NomeCurso", "NomeSerie", "ApelidoTurma", "NomeDisciplina"]:
        if column in frame.columns:
            row[f"{column}_distintos"] = frame[column].nunique(dropna=True)
    coverage_rows.append(row)

pd.DataFrame(coverage_rows).fillna("-")

,base,linhas,periodos_exemplo,NomePeriodo_distintos,NomeCurso_distintos,NomeSerie_distintos,ApelidoTurma_distintos,NomeDisciplina_distintos
0,alunos,19661,"1999, 2000, 2001, 2004, 2005, 2006",26,2,8,98,-
1,notas,239678,"2020, 2021, 2022, 2023, 2024, 2025",7,2,7,37,70.0
2,faltas,168051,"2023, 2024, 2025, 2026",4,2,7,18,57.0
3,pagamentos,220090,"2004, 2005, 2006, 2007, 2008, 2009",23,2,8,98,-
4,responsaveis,39256,"1999, 2000, 2001, 2004, 2005, 2006",26,2,8,98,-
5,professores,1624,"2022, 2023, 2024, 2025, 2026",5,2,7,18,66.0


### O que a cobertura temporal parece visualmente

A tabela acima resume quantos períodos, séries, turmas e disciplinas aparecem em cada base. O quadro abaixo foca ainda mais no aspecto temporal, mostrando diretamente quais anos letivos estão presentes em cada CSV.

In [12]:
period_coverage = []
for name, frame in tables.items():
    if frame is None or "NomePeriodo" not in frame.columns:
        continue
    values = sorted(frame["NomePeriodo"].dropna().astype(str).unique().tolist())
    period_coverage.append({
        "base": name,
        "periodos_detectados": ", ".join(values[:10]),
        "quantidade_periodos": len(values),
    })

pd.DataFrame(period_coverage)

,base,periodos_detectados,quantidade_periodos
0,alunos,"1999, 2000, 2001, 2004, 2005, 2006, 2007, 2008, 2009, 2010",26
1,notas,"2020, 2021, 2022, 2023, 2024, 2025, 2026",7
2,faltas,"2023, 2024, 2025, 2026",4
3,pagamentos,"2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013",23
4,responsaveis,"1999, 2000, 2001, 2004, 2005, 2006, 2007, 2008, 2009, 2010",26
5,professores,"2022, 2023, 2024, 2025, 2026",5


### O que a cobertura temporal revela

Depois desta celula, voce passa a saber se existe base suficiente para validacao temporal. Se os periodos estiverem muito concentrados em um unico ano, a fase de Modeling perde qualidade porque treino, validacao e teste ficam pobres ou inviaveis.

Tambem e aqui que aparece uma visao rapida da cobertura academica: quantos cursos, series, turmas e disciplinas estao realmente representados em cada CSV.

## Entendimento da base de notas

A base de notas e a principal fonte da previsao. Na fase seguinte, ela sera ordenada por aluno, ano, disciplina e etapa para permitir a criacao do alvo de proxima nota e dos baselines temporais.

In [13]:
grades = tables.get("notas")
if grades is not None and "ValorMedia" in grades.columns:
    grade_values = pd.to_numeric(grades["ValorMedia"], errors="coerce")
    display(pd.DataFrame({
        "metrica": ["linhas", "alunos", "disciplinas", "menor_nota", "media", "maior_nota"],
        "valor": [
            len(grades),
            grades["IDAluno"].nunique() if "IDAluno" in grades.columns else None,
            grades["NomeDisciplina"].nunique() if "NomeDisciplina" in grades.columns else None,
            grade_values.min(),
            grade_values.mean(),
            grade_values.max(),
        ],
    }))

    if "NomePeriodo" in grades.columns:
        display(grades.groupby("NomePeriodo", dropna=False).size().reset_index(name="registros_de_nota"))

,metrica,valor
0,linhas,239678.000000
1,alunos,1524.000000
2,disciplinas,70.000000
3,menor_nota,0.000000
4,media,7.795279
5,maior_nota,10.000000


,NomePeriodo,registros_de_nota
0,2020,19299
1,2021,23593
2,2022,25310
3,2023,47072
4,2024,57462
5,2025,54473
6,2026,12469


### Como a base de notas fica visível apos a leitura

Aqui o notebook já mostra algo muito próximo do estilo do GitHub de referência: uma pequena amostra estatística e uma distribuição concreta por período. Isso ajuda a perceber que o dado de notas já chega com volume suficiente para virar série temporal.

### Exemplo visual fixo da base de notas

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes pseudonimizados da base pública:

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | ValorMedia |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 1º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 2º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 3º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 4º BIMESTRE | 7.9 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 1º BIMESTRE | 8.1 |

In [ ]:
grade_sample = public_preview(
    grades,
    rows=5,
    max_columns=8,
)

grade_sample

A amostra acima mostra como uma linha típica da base de notas se apresenta depois da leitura, já pronta para ser entendida na fase seguinte.

### O que a base de notas mostra apos a leitura

A base de notas passa a revelar o coracao do problema. Quando voce executa a celula acima, voce deixa de ver apenas colunas e passa a enxergar a distribuicao real do historico escolar: quantos registros existem, quantos alunos aparecem e como isso se espalha pelos periodos.

Se essa base estiver fraca ou mal distribuida, toda a previsao de proxima nota fica comprometida, porque e dela que nascem os alvos supervisionados.

## Entendimento da base de faltas

As faltas entram como sinal complementar. A pipeline agrega eventos por aluno, ano, disciplina e etapa para criar `faltas_etapa` e `faltas_acumuladas`.

In [15]:
absences = tables.get("faltas")
if absences is not None:
    summary = {
        "eventos_de_falta": len(absences),
        "alunos_com_falta": absences["IDAluno"].nunique() if "IDAluno" in absences.columns else None,
        "disciplinas_com_falta": absences["NomeDisciplina"].nunique() if "NomeDisciplina" in absences.columns else None,
    }
    display(pd.DataFrame([summary]))

    group_cols = [column for column in ["NomePeriodo", "NomeSerie"] if column in absences.columns]
    if group_cols:
        display(absences.groupby(group_cols, dropna=False).size().reset_index(name="faltas").head(20))

,eventos_de_falta,alunos_com_falta,disciplinas_com_falta
0,168051,989,57


,NomePeriodo,NomeSerie,faltas
0,2023,1ª Série,9078
1,2023,2ª Série,7108
2,2023,3ª Série,5546
3,2023,6º Ano,3077
4,2023,7º Ano,3092
5,2023,8º Ano,4566
6,2023,9º Ano,5682
7,2024,1ª Série,14102
8,2024,2ª Série,12543
9,2024,3ª Série,6199


### Como a base de faltas aparece visualmente

A saída acima resume o volume agregado. A amostra abaixo mostra como um evento de falta aparece no dado bruto antes de virar atributo de modelagem.

### Exemplo visual fixo da base de faltas

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes pseudonimizados da base pública:

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | DataFalta |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 3º BIMESTRE | 2025-08-14 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-09 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-13 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-23 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 3º BIMESTRE | 2025-08-12 |

In [ ]:
if absences is not None and not absences.empty:
    absence_sample = public_preview(absences, rows=5, max_columns=8)
    display(absence_sample)

### O que muda quando as faltas entram na leitura

A partir desta saida, faltas deixam de ser apenas eventos soltos e passam a ser percebidas como um sinal que podera ser agregado mais tarde por aluno, periodo, disciplina e etapa.

O que se observa aqui e se existe massa critica para produzir atributos como `faltas_etapa` e `faltas_acumuladas` sem distorcer a analise temporal.

## Entendimento da base de pagamentos

Os pagamentos nao determinam sozinhos o desempenho academico, mas podem funcionar como contexto. Na pipeline eles sao agregados por aluno e ano letivo, usando tambem colunas de datas e flags financeiras para derivar indicadores como pendencia, proporcao de mensalidades e dias ate pagamento, sem expor detalhes individualizados nos relatorios finais.

In [17]:
payments = tables.get("pagamentos")
if payments is not None:
    summary = {
        "movimentos_financeiros": len(payments),
        "alunos_com_movimento": payments["IDAluno"].nunique() if "IDAluno" in payments.columns else None,
        "coluna_status_pagamento": "PagoMovimento" in payments.columns,
        "coluna_mensalidade": "EhMensalidadeMovimento" in payments.columns,
    }
    display(pd.DataFrame([summary]))

    if "PagoMovimento" in payments.columns:
        display(payments["PagoMovimento"].astype(str).value_counts(dropna=False).reset_index(name="quantidade"))

,movimentos_financeiros,alunos_com_movimento,coluna_status_pagamento,coluna_mensalidade
0,220090,6402,True,True


,PagoMovimento,quantidade
0,True,193132
1,False,26958


### Como a base de pagamentos aparece visualmente

Depois da contagem de status, a amostra abaixo mostra o formato de uma linha financeira real. Isso ajuda a entender que o dado ainda está transacional nesta fase e só será consolidado mais adiante.

### Exemplo visual fixo da base de pagamentos

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes pseudonimizados da base pública:

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | DescricaoMovimento | PagoMovimento | EhMensalidadeMovimento | ValorMovimento | ValorPagoMovimento |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - Lights, camera, action - 2o SEMESTRE | True | False | 65.0 | 65.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - 3o bim - Acoragem de ser imperfeito | True | False | 45.0 | 45.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Kit Integrado 6º ano EF | True | False | 2820.0 | 2600.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Agenda 2025 | True | False | 45.0 | 45.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - 4o bim - Tempos e Jeitos para viver | True | False | 45.0 | 45.0 |

In [ ]:
if payments is not None and not payments.empty:
    payment_sample = public_preview(payments, rows=5, max_columns=8)
    display(payment_sample)

### O que a leitura financeira acrescenta

Depois desta celula, a base de pagamentos deixa de parecer apenas administrativa e ganha papel analitico. Ela nao define sozinha o desempenho escolar, mas passa a funcionar como contexto para o risco.

Didaticamente, esta etapa responde: existe sinal financeiro suficiente para virar atributo agregado depois, ou a base esta incompleta demais para isso?

## Entendimento da base de professores

A base de professores permite explicar risco por turma, disciplina e docente. Turmas ou disciplinas sem professor vinculado nao devem ser excluidas; a pipeline usa a categoria `Professor nao identificado ou nao vinculado` nesses casos. Quando ha mais de um professor na mesma combinacao, a preparacao consolida os nomes e registra a quantidade de docentes envolvidos.

In [ ]:
teachers = tables.get("professores")
if teachers is not None:
    summary = {
        "vinculos_professor_disciplina": len(teachers),
        "professores_distintos": teachers["IDFuncionario"].nunique() if "IDFuncionario" in teachers.columns else None,
        "disciplinas_com_professor": teachers["NomeDisciplina"].nunique() if "NomeDisciplina" in teachers.columns else None,
        "turmas_com_professor": teachers["IDTurma"].nunique() if "IDTurma" in teachers.columns else None,
    }
    display(pd.DataFrame([summary]))

### Como a base de professores aparece visualmente

O resumo numérico acima mostra quantos vínculos existem. A amostra abaixo mostra o formato de um vínculo pedagógico antes da consolidação usada na preparação.

### Exemplo visual fixo da base de professores

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes pseudonimizados da base pública:

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeDisciplina | NomeFuncionario |
| --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Arte | Professor 00207 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Ciências da Natureza | Professor 00071 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Educação Física | Professor 00205 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Filosofia | Professor 00091 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Geografia | Professor 00167 |

In [ ]:
if teachers is not None and not teachers.empty:
    teacher_sample = public_preview(teachers, rows=5, max_columns=8)
    display(teacher_sample)

### Como ler a base de professores

Esta saida mostra se o projeto conseguira explicar risco por docente, turma e disciplina. Se houver poucos vinculos ou muitos vazios, o deployment ainda podera funcionar, mas com maior frequencia de casos marcados como professor nao identificado ou nao vinculado.

Isso e importante porque evita descartar turmas inteiras so porque o banco nao informou o professor em todos os casos.

## Verificacao das chaves de integracao

A construcao do dataset final de modelagem depende principalmente destas chaves e colunas operacionais:

- `IDAluno`
- `NomePeriodo`
- `NomeDisciplina`
- `NomeEtapa`
- `IDUnidade`, `IDPeriodo`, `IDCurso`, `IDSerie`, `IDTurma`, `IDDisciplina` para professor

`aluno.csv` e `responsaveis_aluno.csv` permanecem importantes para contexto e relatorios, mas nao sao a espinha dorsal da montagem do dataset supervisionado atual.

O objetivo desta verificacao e encontrar colunas ausentes antes da preparacao dos dados.

### Exemplo visual fixo da verificacao das chaves

| ligacao | chave | objetivo |
| --- | --- | --- |
| notas + faltas | IDAluno + NomePeriodo + NomeDisciplina + stage_order | somar faltas por etapa |
| notas + pagamentos | IDAluno + NomePeriodo | trazer contexto financeiro anual |
| notas + professores | IDTurma + IDDisciplina | atribuir professor quando houver vinculo |

In [21]:
required_columns = {
    "notas": ["IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "ValorMedia"],
    "faltas": ["IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "DataFalta"],
    "pagamentos": ["IDAluno", "NomePeriodo", "PagoMovimento", "EhMensalidadeMovimento", "ValorMovimento", "ValorPagoMovimento", "DataAntecipadoMovimento", "DataVencimentoMovimento"],
    "professores": ["IDUnidade", "IDPeriodo", "IDCurso", "IDSerie", "IDTurma", "IDDisciplina", "IDFuncionario", "NomeFuncionario"],
}

checks = []
for name, required in required_columns.items():
    frame = tables.get(name)
    available = set() if frame is None else set(frame.columns)
    for column in required:
        checks.append({
            "base": name,
            "coluna": column,
            "presente": column in available,
        })

pd.DataFrame(checks)

,base,coluna,presente
0,notas,IDAluno,True
1,notas,NomePeriodo,True
2,notas,NomeDisciplina,True
3,notas,NomeEtapa,True
4,notas,ValorMedia,True
5,faltas,IDAluno,True
6,faltas,NomePeriodo,True
7,faltas,NomeDisciplina,True
8,faltas,NomeEtapa,True
9,faltas,DataFalta,True


### Saída final desta fase de entendimento

Depois desta verificação, o notebook entrega uma resposta visual simples: quais colunas-chave existem e quais faltam. O quadro abaixo resume essa leitura por base.

In [22]:
check_summary = (
    pd.DataFrame(checks)
    .groupby("base", dropna=False)["presente"]
    .agg(total_colunas_verificadas="size", colunas_presentes="sum")
    .reset_index()
)
check_summary["colunas_ausentes"] = check_summary["total_colunas_verificadas"] - check_summary["colunas_presentes"]
check_summary

,base,total_colunas_verificadas,colunas_presentes,colunas_ausentes
0,faltas,5,5,0
1,notas,5,5,0
2,pagamentos,8,8,0
3,professores,8,8,0


Quando este bloco fecha, a fase de Data Understanding já mostrou não só o código de leitura, mas também o aspecto visual do dado e o estado em que cada base chegou para a pipeline.

### O que esta verificacao entrega

Ao final desta checagem, voce sabe se as bases centrais possuem as chaves minimas para serem unidas sem improviso. Se uma coluna obrigatoria estiver faltando, o problema precisa ser resolvido antes da modelagem.

Em outras palavras: aqui termina a fase em que ainda estamos entendendo o dado. A partir daqui, a proxima etapa ja passa a transformá-lo.

## Primeiras conclusoes desta fase

Ao final do Data Understanding, espera-se responder:

- Os CSVs minimos existem?
- Ha anos letivos suficientes para validacao temporal?
- As bases centrais possuem as chaves e colunas necessarias para integracao?
- Existem campos ausentes que precisam de tratamento?
- Ha turmas, disciplinas ou professores com cobertura incompleta?
- A base de notas possui sequencias suficientes para prever a proxima etapa?

Essas respostas orientam a fase seguinte, Data Preparation, onde as bases serao normalizadas, integradas e transformadas em dataset temporal de modelagem.